## Use LangChain's built-in tools and understand how tool calling works

In [1]:
# Install (if needed)
# pip install langchain langchain-community

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.tools import Tool

# ------------------------------------------------------------
# Step 1: Initialize Wikipedia tool
# ------------------------------------------------------------
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

wikipedia_tool = Tool(
    name="Wikipedia",
    func=wiki.run,
    description="Useful for fetching facts from Wikipedia"
)

# ------------------------------------------------------------
# Step 2: Simple Calculator tool
# ------------------------------------------------------------
calculator_tool = Tool(
    name="Calculator",
    func=lambda x: eval(x),
    description="Useful for running math expressions like '3+12'"
)

# ------------------------------------------------------------
# Step 3: Call tools directly (no agent)
# ------------------------------------------------------------
query_1 = "Machine learning"
result_1 = wikipedia_tool.run(query_1)

query_2 = "12 * 7 + 5"
result_2 = calculator_tool.run(query_2)

# ------------------------------------------------------------
# Step 4: Print results
# ------------------------------------------------------------
print("\nWikipedia Tool Result:\n")
print(result_1)

print("\nCalculator Tool Result:\n")
print(result_2)



Wikipedia Tool Result:

Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalise to unseen data, and thus perform tasks without explicit instructions. Within a subdiscipline in machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
ML finds application in many fields, including natural language processing, computer vision, speech recognition, email filtering, agriculture, and medicine. The application of ML to business problems is known as predictive analytics.
Statistics and mathematical optimisation (mathematical programming) methods comprise the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) via unsupervised learning.
From a theor

## Build a simple agent using a prompt template + 2 tools (e.g., calculator + search)

In [ ]:
# Import libraries
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.prompts import PromptTemplate
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

# Step 1: Initialize the LLM
llm = ChatOpenAI(temperature=0)

# Step 2: Load tools (calculator + wikipedia)
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
tools = [
    Tool(
        name="Calculator",
        func=lambda x: eval(x),  # quick demo; better use llm-math tool
        description="Useful for doing math calculations"
    ),
    Tool(
        name="Wikipedia",
        func=wikipedia.run,
        description="Useful for searching facts from Wikipedia"
    )
]

# Step 3: Define a custom prompt template
template = """You are a helpful assistant that can use tools.
Always explain briefly before giving the final answer.
If math is required, use the calculator tool.
If external information is required, use the Wikipedia tool.
Question: {question}
"""

prompt = PromptTemplate(
    input_variables=["question"],
    template=template
)

# Step 4: Initialize the agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Step 5: Run a sample query
query = "What is the population of France?"
formatted_prompt = prompt.format(question=query)

result = agent.run(formatted_prompt)
print("\nFinal Answer:", result)
